In [ ]:
!pip install ultralytics opencv-python


In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO('yolov8n.pt')

In [ ]:
from google.colab import files

In [ ]:
!pip install ultralytics supervision

In [ ]:
!pip install requests

In [ ]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO
import time
from collections import defaultdict
from google.colab.patches import cv2_imshow

class WrongSideDrivingDetector:
    def __init__(self, model_path='yolov8n.pt', confidence=0.5, lane_direction='right'):
        """
        Initialize the wrong-side driving detector.

        Args:
            model_path: Path to YOLOv8 model file
            confidence: Detection confidence threshold
            lane_direction: Expected driving direction ('left' or 'right')
        """
        self.model = YOLO(model_path)
        self.confidence = confidence
        self.lane_direction = lane_direction

        # Vehicle classes in COCO dataset
        self.vehicle_classes = [2, 3, 5, 7]  # car, motorcycle, bus, truck

        # Direction tracking
        self.object_tracks = defaultdict(list)
        self.violation_detected = False
        self.violation_count = 0

        # Define lane regions (adjust based on your specific road layout)
        self.setup_lanes = False

    def define_lanes(self, frame):
        """Define lane regions based on the first frame."""
        height, width = frame.shape[:2]

        # For a typical two-way road
        if self.lane_direction == 'right':
            # In right-driving countries, vehicles drive on the right side
            self.correct_direction_lane = {'x_min': 0, 'x_max': width // 2, 'y_min': 0, 'y_max': height}
            self.wrong_direction_lane = {'x_min': width // 2, 'x_max': width, 'y_min': 0, 'y_max': height}
        else:
            # In left-driving countries, vehicles drive on the left side
            self.correct_direction_lane = {'x_min': width // 2, 'x_max': width, 'y_min': 0, 'y_max': height}
            self.wrong_direction_lane = {'x_min': 0, 'x_max': width // 2, 'y_min': 0, 'y_max': height}

        self.setup_lanes = True

    def calculate_center(self, box):
        """Calculate the center point of a bounding box."""
        x1, y1, x2, y2 = box
        return ((x1 + x2) // 2, (y1 + y2) // 2)

    def determine_direction(self, track, min_points=5):
        """Determine the direction of movement based on the object's track."""
        if len(track) < min_points:
            return None

        # Get first and last points
        first_x, first_y = track[0]
        last_x, last_y = track[-1]

        # Calculate x-direction (left-to-right or right-to-left)
        if last_x - first_x > 20:  # Moving right
            return "right"
        elif first_x - last_x > 20:  # Moving left
            return "left"
        else:
            return None  # Not enough horizontal movement to determine

    def is_in_wrong_lane(self, center, direction):
        """Check if the object is in the wrong lane based on its direction."""
        x, y = center

        if not direction:
            return False

        if self.lane_direction == 'right':
            # For right-driving countries
            if (direction == "left" and
                x > self.wrong_direction_lane['x_min'] and
                x < self.wrong_direction_lane['x_max']):
                return False  # Correct direction in the right lane
            elif (direction == "right" and
                  x > self.correct_direction_lane['x_min'] and
                  x < self.correct_direction_lane['x_max']):
                return False  # Correct direction in the left lane
            else:
                return True  # Wrong direction
        else:
            # For left-driving countries
            if (direction == "right" and
                x > self.wrong_direction_lane['x_min'] and
                x < self.wrong_direction_lane['x_max']):
                return False  # Correct direction in the left lane
            elif (direction == "left" and
                  x > self.correct_direction_lane['x_min'] and
                  x < self.correct_direction_lane['x_max']):
                return False  # Correct direction in the right lane
            else:
                return True  # Wrong direction

    def process_video(self, video_path, output_path=None, display=True):
        """
        Process a video to detect wrong-side driving.

        Args:
            video_path: Path to the input video
            output_path: Path to save the output video (optional)
            display: Whether to display the processed frames

        Returns:
            bool: Whether wrong-side driving was detected
        """
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Error opening video file {video_path}")
            return False

        # Get video properties
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)

        # Initialize video writer if output path is provided
        writer = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1

            # Define lane regions based on the first frame
            if not self.setup_lanes:
                self.define_lanes(frame)

            # Draw lane divider
            cv2.line(frame, (width // 2, 0), (width // 2, height), (255, 255, 0), 2)

            # Perform detection
            results = self.model(frame, verbose=False)[0]

            # Process detections
            for r in results.boxes.data.tolist():
                x1, y1, x2, y2, conf, cls = r

                # Filter by confidence and class
                if conf < self.confidence or int(cls) not in self.vehicle_classes:
                    continue

                # Generate unique ID based on initial position
                if frame_count == 1:
                    obj_id = f"{int(x1)}_{int(y1)}"
                    self.object_tracks[obj_id].append(self.calculate_center((x1, y1, x2, y2)))
                else:
                    # Find closest track
                    center = self.calculate_center((x1, y1, x2, y2))
                    min_dist = float('inf')
                    closest_id = None

                    for obj_id, track in self.object_tracks.items():
                        if track:  # If track is not empty
                            last_center = track[-1]
                            dist = np.sqrt((center[0] - last_center[0])**2 + (center[1] - last_center[1])**2)
                            if dist < min_dist and dist < 100:  # Threshold for considering it the same object
                                min_dist = dist
                                closest_id = obj_id

                    if closest_id:
                        self.object_tracks[closest_id].append(center)
                    else:
                        new_id = f"{int(x1)}_{int(y1)}_{frame_count}"
                        self.object_tracks[new_id].append(center)

                # Draw bounding box
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)

                # Process each track to detect wrong-side driving
                for obj_id, track in self.object_tracks.items():
                    if len(track) >= 5:  # Only process tracks with enough points
                        direction = self.determine_direction(track)
                        if direction:
                            # Check if current position is in wrong lane
                            current_pos = track[-1]
                            wrong_lane = self.is_in_wrong_lane(current_pos, direction)

                            if wrong_lane:
                                self.violation_detected = True
                                self.violation_count += 1

                                # Draw violation indicator
                                cv2.putText(frame, "WRONG-SIDE DRIVING!", (50, 50),
                                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

                                # Draw movement path
                                for i in range(1, len(track)):
                                    cv2.line(frame,
                                           (int(track[i-1][0]), int(track[i-1][1])),
                                           (int(track[i][0]), int(track[i][1])),
                                           (0, 0, 255), 2)

            # Display lane direction information
            lane_info = "Lane Direction: Left-side driving" if self.lane_direction == 'left' else "Lane Direction: Right-side driving"
            cv2.putText(frame, lane_info, (width - 400, height - 20),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

            # Display violation count
            cv2.putText(frame, f"Violations: {self.violation_count}", (width - 200, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            # Write frame to output video
            if writer:
                writer.write(frame)

            # Display frame
            if display:
                cv2_imshow(frame)  # Changed from cv2.imshow to cv2_imshow
                # Removed cv2.waitKey as it's not needed with cv2_imshow

            # Add a small delay to process events
            time.sleep(0.01)

        # Release resources
        cap.release()
        if writer:
            writer.release()
        cv2.destroyAllWindows()

        return self.violation_detected, self.violation_count

def main():
    # Create detector instance
    detector = WrongSideDrivingDetector(
        model_path='yolov8n.pt',  # Use yolov8n.pt or your custom trained model
        confidence=0.5,
        lane_direction='right'  # Set based on your country's driving convention
    )

    # Process video
    video_path = '/content/traffic2.mp4'  # Replace with your video file
    output_path = 'output_video.mp4'  # Output video with detections

    violation_detected, violation_count = detector.process_video(
        video_path=video_path,
        output_path=output_path,
        display=True
    )

    # Print results
    if violation_detected:
        print(f"ALERT: Wrong-side driving detected! {violation_count} violations.")
    else:
        print("No wrong-side driving detected.")

if __name__ == "__main__":
    main()

In [ ]:
# Create a new cell in your Colab notebook and run this
from google.colab import files
import os

# Upload files
uploaded = files.upload()

# First uploaded file name
file_name = next(iter(uploaded))
print(f"Uploaded file: {file_name}")

# Install required packages if not already installed
!pip install ultralytics opencv-python pytesseract

# For Tesseract OCR in Colab
!apt-get install -y tesseract-ocr

In [ ]:
# shaivi red light
import cv2
import numpy as np
import time
from ultralytics import YOLO
import argparse

def parse_arguments():
    """Parse command line arguments."""
    parser = argparse.ArgumentParser(description='Red Light Violation Detection')
    parser.add_argument('--video', type=str, required=True, help='Path to input video file')
    parser.add_argument('--output', type=str, default='output.mp4', help='Path to output video file')
    parser.add_argument('--confidence', type=float, default=0.5, help='Minimum confidence for object detection')
    return parser.parse_args()

class RedLightViolationDetector:
    def __init__(self, video_path, confidence=0.5):
        """Initialize the red light violation detector."""
        self.video_path = video_path
        self.confidence = confidence
        self.model = YOLO('yolov8n.pt')  # Load YOLOv8 model (you can use larger models like 'm', 'l', 'x')
        self.cap = cv2.VideoCapture(video_path)
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Traffic light state
        self.traffic_light_state = "red"  # Initialize as red
        self.last_state_change = time.time()
        self.state_duration = {"red": 30, "yellow": 5, "green": 30}  # Duration in seconds

        # Define stop line (customize based on your video)
        self.stop_line_y = int(self.height * 0.6)  # Adjust this value based on your camera view

        # Track vehicles that have already violated
        self.violators = set()
        self.next_violator_id = 1

        # For tracking vehicles across frames
        self.tracked_vehicles = {}
        self.last_vehicle_id = 0

        # Initialize traffic light position (customize based on your video)
        self.traffic_light_roi = [
            int(self.width * 0.7),  # x-coordinate
            int(self.height * 0.2),  # y-coordinate
            int(self.width * 0.1),   # width
            int(self.height * 0.1)    # height
        ]

    def detect_traffic_light(self, frame):
        """
        Detect the state of the traffic light.
        In a real implementation, this would use computer vision to detect the traffic light color.
        Here we simulate changing light states based on time.
        """
        # For a real implementation, you would analyze the traffic light ROI
        # and detect the active light color using HSV color filtering

        # Simulated traffic light state based on time
        elapsed_time = time.time() - self.last_state_change
        current_duration = self.state_duration[self.traffic_light_state]

        if elapsed_time > current_duration:
            self.last_state_change = time.time()
            if self.traffic_light_state == "red":
                self.traffic_light_state = "green"
            elif self.traffic_light_state == "green":
                self.traffic_light_state = "yellow"
            elif self.traffic_light_state == "yellow":
                self.traffic_light_state = "red"

        return self.traffic_light_state

    def detect_vehicles(self, frame):
        """Detect vehicles in the frame using YOLOv8."""
        vehicle_classes = [2, 3, 5, 7]  # COCO classes for car, motorbike, bus, truck
        results = self.model(frame, verbose=False, conf=self.confidence)

        vehicles = []
        for result in results:
            boxes = result.boxes.cpu().numpy()
            for box in boxes:
                if int(box.cls[0]) in vehicle_classes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    confidence = float(box.conf[0])
                    class_id = int(box.cls[0])
                    vehicles.append({
                        "bbox": (x1, y1, x2, y2),
                        "confidence": confidence,
                        "class_id": class_id
                    })

        return vehicles

    def track_vehicles(self, vehicles):
        """
        Simple vehicle tracking across frames using IoU.
        This is a basic implementation - real tracking would use more sophisticated algorithms.
        """
        if not self.tracked_vehicles:
            # First frame
            for vehicle in vehicles:
                self.last_vehicle_id += 1
                self.tracked_vehicles[self.last_vehicle_id] = {
                    "bbox": vehicle["bbox"],
                    "crossed_line": False,
                    "violated": False,
                    "frames_since_update": 0
                }
            return

        # Match current detections with existing tracks
        matched_tracks = {}
        unmatched_detections = []

        # Mark all existing tracks for update checking
        for track_id in self.tracked_vehicles:
            self.tracked_vehicles[track_id]["frames_since_update"] += 1

        for vehicle in vehicles:
            best_iou = 0.3  # IOU threshold
            best_id = -1

            for track_id, track in self.tracked_vehicles.items():
                iou = self.calculate_iou(vehicle["bbox"], track["bbox"])
                if iou > best_iou:
                    best_iou = iou
                    best_id = track_id

            if best_id != -1:
                # Update existing track
                self.tracked_vehicles[best_id]["bbox"] = vehicle["bbox"]
                self.tracked_vehicles[best_id]["frames_since_update"] = 0
                matched_tracks[best_id] = True
            else:
                # New track
                unmatched_detections.append(vehicle)

        # Add new tracks
        for vehicle in unmatched_detections:
            self.last_vehicle_id += 1
            self.tracked_vehicles[self.last_vehicle_id] = {
                "bbox": vehicle["bbox"],
                "crossed_line": False,
                "violated": False,
                "frames_since_update": 0
            }

        # Remove stale tracks
        track_ids = list(self.tracked_vehicles.keys())
        for track_id in track_ids:
            if self.tracked_vehicles[track_id]["frames_since_update"] > 15:  # Remove after 15 frames of no updates
                del self.tracked_vehicles[track_id]

    def calculate_iou(self, box1, box2):
        """Calculate IoU between two bounding boxes."""
        x1_1, y1_1, x2_1, y2_1 = box1
        x1_2, y1_2, x2_2, y2_2 = box2

        # Calculate the coordinates of the intersection rectangle
        x_left = max(x1_1, x1_2)
        y_top = max(y1_1, y1_2)
        x_right = min(x2_1, x2_2)
        y_bottom = min(y2_1, y2_2)

        # No intersection
        if x_right < x_left or y_bottom < y_top:
            return 0.0

        # Calculate area of intersection
        intersection_area = (x_right - x_left) * (y_bottom - y_top)

        # Calculate area of both bounding boxes
        box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
        box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)

        # Calculate IoU
        iou = intersection_area / float(box1_area + box2_area - intersection_area)
        return iou

    def detect_violations(self, traffic_light_state):
        """Detect red light violations."""
        violations = []

        if traffic_light_state == "red":
            for track_id, vehicle in self.tracked_vehicles.items():
                x1, y1, x2, y2 = vehicle["bbox"]
                vehicle_bottom_y = y2

                # Check if vehicle crossed the stop line while the light is red
                if vehicle_bottom_y > self.stop_line_y and not vehicle["crossed_line"]:
                    vehicle["crossed_line"] = True
                    vehicle["violated"] = True
                    if track_id not in self.violators:
                        violations.append({
                            "id": self.next_violator_id,
                            "bbox": vehicle["bbox"],
                            "timestamp": time.time()
                        })
                        self.violators.add(track_id)
                        self.next_violator_id += 1
        else:
            # Reset crossed_line flag when light is not red
            for track_id, vehicle in self.tracked_vehicles.items():
                vehicle["crossed_line"] = False

        return violations

    def process_video(self, output_path):
        """Process the video and detect red light violations."""
        # Define codec and create VideoWriter object
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, self.fps, (self.width, self.height))

        violation_count = 0
        frame_count = 0

        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret:
                break

            frame_count += 1
            if frame_count % 3 != 0:  # Process every 3rd frame for efficiency
                continue

            # Detect traffic light state
            traffic_light_state = self.detect_traffic_light(frame)

            # Detect vehicles
            vehicles = self.detect_vehicles(frame)

            # Track vehicles
            self.track_vehicles(vehicles)

            # Detect violations
            new_violations = self.detect_violations(traffic_light_state)
            violation_count += len(new_violations)

            # Draw results on frame
            self.draw_results(frame, traffic_light_state, new_violations)

            # Write the frame
            out.write(frame)

            # Show the frame (comment out for faster processing)
            cv2.imshow('Red Light Violation Detection', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        self.cap.release()
        out.release()
        cv2.destroyAllWindows()

        print(f"Processing complete. Detected {violation_count} violations.")
        return violation_count

    def draw_results(self, frame, traffic_light_state, new_violations):
        """Draw bounding boxes, stop line, and violation information on the frame."""
        # Draw stop line
        cv2.line(frame, (0, self.stop_line_y), (self.width, self.stop_line_y), (255, 0, 0), 2)

        # Draw traffic light indicator
        light_color = (0, 0, 255) if traffic_light_state == "red" else \
                     (0, 255, 255) if traffic_light_state == "yellow" else \
                     (0, 255, 0)
        cv2.rectangle(frame,
                     (self.traffic_light_roi[0], self.traffic_light_roi[1]),
                     (self.traffic_light_roi[0] + self.traffic_light_roi[2],
                      self.traffic_light_roi[1] + self.traffic_light_roi[3]),
                     light_color, -1)
        cv2.putText(frame, traffic_light_state.upper(),
                   (self.traffic_light_roi[0], self.traffic_light_roi[1] - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, light_color, 2)

        # Draw tracked vehicles
        for track_id, vehicle in self.tracked_vehicles.items():
            x1, y1, x2, y2 = vehicle["bbox"]
            color = (0, 0, 255) if vehicle["violated"] else (255, 0, 0)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            if vehicle["violated"]:
                cv2.putText(frame, f"VIOLATION", (x1, y1 - 10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        # Draw new violations
        for violation in new_violations:
            x1, y1, x2, y2 = violation["bbox"]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cv2.putText(frame, f"VIOLATION #{violation['id']}",
                       (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # Display status information
        cv2.putText(frame, f"Light: {traffic_light_state.upper()}",
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Total Violations: {len(self.violators)}",
                   (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

def main():
    """Main function."""
    args = parse_arguments()

    detector = RedLightViolationDetector(
        video_path=args.video,
        confidence=args.confidence
    )

    violation_count = detector.process_video(args.output)
    print(f"Total violations detected: {violation_count}")

if __name__ == "__main__":
    main()

In [ ]:
# Traffic Violation Detection System
# - Red-light violation detection
# - Wrong-side driving detection
# - Basic crash detection (proximity-based)
# - Violation screenshots
# - Violation logging

# Install required packages (run this first)
!pip install ultralytics opencv-python

import cv2
import numpy as np
import torch
import os
import time
from datetime import datetime
from ultralytics import YOLO
from google.colab import files
import matplotlib.pyplot as plt
from IPython.display import display, HTML

class TrafficViolationDetector:
    def __init__(self, video_path, output_dir="./violation_results"):
        """Initialize the traffic violation detector.

        Args:
            video_path: Path to input video file
            output_dir: Directory to save violation screenshots and logs
        """
        self.video_path = video_path
        self.output_dir = output_dir
        self.screenshot_dir = f"{output_dir}/screenshots"
        self.log_file = f"{output_dir}/violations_log.txt"

        # Create output directories
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.screenshot_dir, exist_ok=True)

        # Initialize violation log
        with open(self.log_file, 'w') as f:
            f.write("Timestamp,Violation Type,Vehicle ID,Confidence\n")

        # Load YOLOv8 model
        self.model = YOLO('yolov8n.pt')

        # Check for GPU
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {self.device}")

        # Initialize tracker
        self.tracks = {}
        self.next_id = 0
        self.max_age = 5  # Track aging (5-frame buffer)

        # Initialize traffic light state
        self.light_state = "unknown"  # Can be "red", "yellow", "green", or "unknown"
        self.light_state_buffer = []

        # Set up video properties
        self.cap = cv2.VideoCapture(video_path)
        self.frame_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.frame_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.fps = int(self.cap.get(cv2.CAP_PROP_FPS))

        # Define regions of interest
        self.traffic_light_roi = [0, 0, self.frame_width, int(self.frame_height * 0.4)]

        # Define parameters for violation detection
        self.stop_line_y = int(self.frame_height * 0.6)  # Position of the stop line
        self.lane_divider_x = int(self.frame_width * 0.5)  # Center line dividing lanes
        self.crash_distance_threshold = 50  # Pixel distance threshold for crash detection

        # Store traffic direction (will be determined from initial frames)
        self.traffic_direction = None
        self.direction_samples = []

        # Lists to store current violations
        self.red_light_violations = set()
        self.wrong_side_violations = set()
        self.crash_violations = set()

        print(f"Initialized Traffic Violation Detector for {video_path}")
        print(f"Video dimensions: {self.frame_width}x{self.frame_height}, FPS: {self.fps}")

    def detect_traffic_light(self, frame):
        """Detect traffic lights in the frame and determine their state."""
        # Extract ROI for traffic light detection
        roi = frame[
            self.traffic_light_roi[1]:self.traffic_light_roi[3],
            self.traffic_light_roi[0]:self.traffic_light_roi[2]
        ]

        # Detect all objects in the ROI
        results = self.model(roi, classes=[9], conf=0.3)  # Class 9 is traffic light in COCO

        traffic_lights = []
        for r in results:
            boxes = r.boxes
            for box in boxes:
                # Get box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                # Convert back to original frame coordinates
                x1 += self.traffic_light_roi[0]
                y1 += self.traffic_light_roi[1]
                x2 += self.traffic_light_roi[0]
                y2 += self.traffic_light_roi[1]

                # Extract traffic light region
                light_roi = frame[int(y1):int(y2), int(x1):int(x2)]
                if light_roi.size == 0:
                    continue

                # Analyze traffic light color
                color = self.analyze_traffic_light_color(light_roi)
                traffic_lights.append(color)

        # Update traffic light state based on detected lights
        if traffic_lights:
            # Use most common color if multiple lights detected
            current_state = max(set(traffic_lights), key=traffic_lights.count)

            # Use buffer for stability
            self.light_state_buffer.append(current_state)
            if len(self.light_state_buffer) > 5:
                self.light_state_buffer.pop(0)

            # Use majority voting for final state
            self.light_state = max(set(self.light_state_buffer), key=self.light_state_buffer.count)

        return self.light_state

    def analyze_traffic_light_color(self, roi):
        """Analyze the color of a traffic light ROI."""
        if roi.size == 0:
            return "unknown"

        # Convert to HSV for better color detection
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

        # Define HSV ranges for traffic light colors
        red_lower1 = np.array([0, 100, 100])
        red_upper1 = np.array([10, 255, 255])
        red_lower2 = np.array([160, 100, 100])
        red_upper2 = np.array([180, 255, 255])
        yellow_lower = np.array([20, 100, 100])
        yellow_upper = np.array([35, 255, 255])
        green_lower = np.array([40, 100, 100])
        green_upper = np.array([80, 255, 255])

        # Create masks for each color
        red_mask1 = cv2.inRange(hsv, red_lower1, red_upper1)
        red_mask2 = cv2.inRange(hsv, red_lower2, red_upper2)
        red_mask = cv2.bitwise_or(red_mask1, red_mask2)
        yellow_mask = cv2.inRange(hsv, yellow_lower, yellow_upper)
        green_mask = cv2.inRange(hsv, green_lower, green_upper)

        # Count pixels of each color
        red_pixels = cv2.countNonZero(red_mask)
        yellow_pixels = cv2.countNonZero(yellow_mask)
        green_pixels = cv2.countNonZero(green_mask)

        # Get the dominant color
        max_pixels = max(red_pixels, yellow_pixels, green_pixels)

        if max_pixels < 10:  # Minimum threshold to consider a color detected
            return "unknown"
        elif max_pixels == red_pixels:
            return "red"
        elif max_pixels == yellow_pixels:
            return "yellow"
        else:
            return "green"

    def detect_vehicles(self, frame):
        """Detect vehicles in the frame."""
        # Detect objects (focus on vehicle classes)
        # COCO classes: 2-car, 3-motorcycle, 5-bus, 7-truck
        results = self.model(frame, classes=[2, 3, 5, 7], conf=0.4)

        detections = []
        for r in results:
            boxes = r.boxes
            for box in boxes:
                # Get coordinates and class
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                cls = int(box.cls[0])
                conf = float(box.conf[0])

                # Calculate center point for direction analysis
                center_x = (x1 + x2) / 2
                center_y = (y1 + y2) / 2

                detections.append({
                    'bbox': (x1, y1, x2, y2),
                    'center': (center_x, center_y),
                    'class': cls,
                    'conf': conf
                })

        return detections

    def update_tracks(self, detections):
        """Update vehicle tracks using IoU-based tracking."""
        # If no existing tracks, initialize with detections
        if not self.tracks:
            for det in detections:
                self.tracks[self.next_id] = {
                    'bbox': det['bbox'],
                    'center': det['center'],
                    'class': det['class'],
                    'conf': det['conf'],
                    'age': 0,
                    'history': [det['center']],  # Store position history for direction
                    'direction': None,  # Will be calculated later
                    'moving': False
                }
                self.next_id += 1
            return self.tracks

        # Match detections to existing tracks
        matches = []
        unmatched_tracks = list(self.tracks.keys())
        unmatched_detections = list(range(len(detections)))

        # Calculate IoU for each track-detection pair
        for t_idx in unmatched_tracks:
            for d_idx in unmatched_detections:
                iou = self.calculate_iou(
                    self.tracks[t_idx]['bbox'],
                    detections[d_idx]['bbox']
                )
                if iou > 0.5:  # IoU threshold
                    matches.append((t_idx, d_idx))
                    unmatched_tracks.remove(t_idx)
                    unmatched_detections.remove(d_idx)
                    break

        # Update matched tracks
        for t_idx, d_idx in matches:
            self.tracks[t_idx]['bbox'] = detections[d_idx]['bbox']
            self.tracks[t_idx]['center'] = detections[d_idx]['center']
            self.tracks[t_idx]['conf'] = detections[d_idx]['conf']
            self.tracks[t_idx]['age'] = 0

            # Update position history (keep last 10 positions)
            self.tracks[t_idx]['history'].append(detections[d_idx]['center'])
            if len(self.tracks[t_idx]['history']) > 10:
                self.tracks[t_idx]['history'].pop(0)

            # Calculate direction if we have enough history
            if len(self.tracks[t_idx]['history']) >= 3:
                self.tracks[t_idx]['direction'] = self.calculate_direction(self.tracks[t_idx]['history'])
                self.tracks[t_idx]['moving'] = self.is_moving(self.tracks[t_idx]['history'])

        # Increment age of unmatched tracks
        for t_idx in unmatched_tracks:
            self.tracks[t_idx]['age'] += 1

        # Remove old tracks
        self.tracks = {k: v for k, v in self.tracks.items() if v['age'] < self.max_age}

        # Add new tracks for unmatched detections
        for d_idx in unmatched_detections:
            self.tracks[self.next_id] = {
                'bbox': detections[d_idx]['bbox'],
                'center': detections[d_idx]['center'],
                'class': detections[d_idx]['class'],
                'conf': detections[d_idx]['conf'],
                'age': 0,
                'history': [detections[d_idx]['center']],
                'direction': None,
                'moving': False
            }
            self.next_id += 1

        return self.tracks

    def calculate_iou(self, bbox1, bbox2):
        """Calculate IoU between two bounding boxes."""
        x1_min, y1_min, x1_max, y1_max = bbox1
        x2_min, y2_min, x2_max, y2_max = bbox2

        # Calculate intersection area
        x_min = max(x1_min, x2_min)
        y_min = max(y1_min, y2_min)
        x_max = min(x1_max, x2_max)
        y_max = min(y1_max, y2_max)

        if x_max < x_min or y_max < y_min:
            return 0.0

        intersection = (x_max - x_min) * (y_max - y_min)

        # Calculate areas
        area1 = (x1_max - x1_min) * (y1_max - y1_min)
        area2 = (x2_max - x2_min) * (y2_max - y2_min)

        # Calculate IoU
        iou = intersection / float(area1 + area2 - intersection)
        return iou

    def calculate_direction(self, positions):
        """Calculate the direction of movement based on position history."""
        if len(positions) < 2:
            return None

        # Get the first and last positions
        first_x, first_y = positions[0]
        last_x, last_y = positions[-1]

        # Calculate horizontal and vertical movement
        dx = last_x - first_x
        dy = last_y - first_y

        # Determine primary direction
        if abs(dx) > abs(dy):
            # Horizontal movement is dominant
            return "right" if dx > 0 else "left"
        else:
            # Vertical movement is dominant
            return "down" if dy > 0 else "up"

    def is_moving(self, positions):
        """Determine if the object is moving based on position history."""
        if len(positions) < 3:
            return False

        # Calculate total distance moved
        total_movement = 0
        for i in range(1, len(positions)):
            x1, y1 = positions[i-1]
            x2, y2 = positions[i]
            distance = np.sqrt((x2-x1)**2 + (y2-y1)**2)
            total_movement += distance

        # Consider moving if total movement exceeds threshold
        return total_movement > 10

    def determine_traffic_direction(self):
        """Determine the normal traffic direction based on observed vehicle movements."""
        # Collect direction samples from moving vehicles
        for track_id, track in self.tracks.items():
            if track['moving'] and track['direction'] in ['left', 'right']:
                self.direction_samples.append(track['direction'])

        # Keep only the last 50 samples
        if len(self.direction_samples) > 50:
            self.direction_samples = self.direction_samples[-50:]

        # Determine predominant direction if we have enough samples
        if len(self.direction_samples) >= 20:
            left_count = self.direction_samples.count('left')
            right_count = self.direction_samples.count('right')

            if left_count > right_count * 2:  # If left is at least twice as common
                self.traffic_direction = 'left'
            elif right_count > left_count * 2:  # If right is at least twice as common
                self.traffic_direction = 'right'

    def detect_violations(self, frame):
        """Detect traffic violations based on vehicle tracks and traffic light state."""
        # If traffic direction is not determined yet, try to determine it
        if self.traffic_direction is None:
            self.determine_traffic_direction()

        # Reset current violation lists
        current_red_light = set()
        current_wrong_side = set()
        current_crash = set()

        # Check for red light violations
        if self.light_state == "red":
            for track_id, track in self.tracks.items():
                # Get vehicle position
                _, _, _, y2 = track['bbox']  # Bottom of bounding box

                # Check if vehicle is moving
                if not track['moving']:
                    continue

                # Check if vehicle is crossing stop line during red light
                if y2 > self.stop_line_y and track['direction'] == 'down':
                    current_red_light.add(track_id)

        # Check for wrong side driving
        if self.traffic_direction is not None:
            for track_id, track in self.tracks.items():
                if not track['moving'] or track['direction'] not in ['left', 'right']:
                    continue

                # Get vehicle center position
                center_x, _ = track['center']

                # For left-side driving countries
                if self.traffic_direction == 'left' and track['direction'] == 'right' and center_x < self.lane_divider_x:
                    current_wrong_side.add(track_id)

                # For right-side driving countries
                elif self.traffic_direction == 'right' and track['direction'] == 'left' and center_x > self.lane_divider_x:
                    current_wrong_side.add(track_id)

        # Check for crash detection (proximity-based)
        track_ids = list(self.tracks.keys())
        for i in range(len(track_ids)):
            for j in range(i+1, len(track_ids)):
                track1 = self.tracks[track_ids[i]]
                track2 = self.tracks[track_ids[j]]

                # Calculate distance between vehicles
                x1, y1 = track1['center']
                x2, y2 = track2['center']
                distance = np.sqrt((x2-x1)**2 + (y2-y1)**2)

                # Check if distance is below threshold and both are moving
                if distance < self.crash_distance_threshold and track1['moving'] and track2['moving']:
                    current_crash.add(track_ids[i])
                    current_crash.add(track_ids[j])

        # Check for new violations
        new_red_light = current_red_light - self.red_light_violations
        new_wrong_side = current_wrong_side - self.wrong_side_violations
        new_crash = current_crash - self.crash_violations

        # Handle new violations
        self.handle_new_violations(frame, new_red_light, "Red Light Violation")
        self.handle_new_violations(frame, new_wrong_side, "Wrong Side Driving")
        self.handle_new_violations(frame, new_crash, "Potential Crash")

        # Update current violation sets
        self.red_light_violations = current_red_light
        self.wrong_side_violations = current_wrong_side
        self.crash_violations = current_crash

        # Return all current violations
        return list(current_red_light), list(current_wrong_side), list(current_crash)

    def handle_new_violations(self, frame, violation_ids, violation_type):
        """Handle new violations by taking screenshots and logging."""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        for track_id in violation_ids:
            if track_id in self.tracks:
                # Take screenshot
                file_name = f"{self.screenshot_dir}/{violation_type.replace(' ', '_')}_{track_id}_{timestamp}.jpg"
                cv2.imwrite(file_name, frame)

                # Log violation
                with open(self.log_file, 'a') as f:
                    f.write(f"{timestamp},{violation_type},{track_id},{self.tracks[track_id]['conf']:.2f}\n")

    def draw_annotations(self, frame):
        """Draw annotations on the frame."""
        # Draw stop line
        cv2.line(frame, (0, self.stop_line_y), (self.frame_width, self.stop_line_y), (255, 0, 0), 2)

        # Draw lane divider
        cv2.line(frame, (self.lane_divider_x, 0), (self.lane_divider_x, self.frame_height), (255, 255, 0), 2)

        # Draw traffic light state
        light_color = (0, 0, 255) if self.light_state == "red" else \
                     (0, 255, 255) if self.light_state == "yellow" else \
                     (0, 255, 0) if self.light_state == "green" else (200, 200, 200)
        cv2.putText(frame, f"Light: {self.light_state.upper()}", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, light_color, 2)

        # Draw traffic direction
        if self.traffic_direction:
            cv2.putText(frame, f"Traffic: {self.traffic_direction.upper()}", (10, 70),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        # Draw vehicle tracks and violations
        for track_id, track in self.tracks.items():
            x1, y1, x2, y2 = [int(c) for c in track['bbox']]

            # Determine color based on violations
            color = (0, 255, 0)  # Default green

            # Check if this track has violations
            violation_text = ""
            if track_id in self.red_light_violations:
                color = (0, 0, 255)  # Red for red light violations
                violation_text += "RED LIGHT "

            if track_id in self.wrong_side_violations:
                color = (0, 165, 255)  # Orange for wrong side
                violation_text += "WRONG SIDE "

            if track_id in self.crash_violations:
                color = (128, 0, 128)  # Purple for crash
                violation_text += "CRASH "

            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Draw ID and direction
            if track['direction']:
                cv2.putText(frame, f"ID:{track_id} {track['direction']}",
                           (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            else:
                cv2.putText(frame, f"ID:{track_id}",
                           (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Draw violation text if any
            if violation_text:
                cv2.putText(frame, violation_text,
                           (x1, y2 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

            # Draw movement history
            if len(track['history']) > 1:
                for i in range(1, len(track['history'])):
                    pt1 = (int(track['history'][i-1][0]), int(track['history'][i-1][1]))
                    pt2 = (int(track['history'][i][0]), int(track['history'][i][1]))
                    cv2.line(frame, pt1, pt2, color, 2)

        return frame

    def process_video(self, display_output=False, output_video_path=None):
        """Process the input video and detect violations."""
        frame_count = 0
        start_time = time.time()

        # Create output video writer if needed
        if output_video_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(
                output_video_path,
                fourcc,
                self.fps,
                (self.frame_width, self.frame_height)
            )

        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret:
                break

            frame_count += 1

            # Skip every other frame for faster processing (optional)
            if frame_count % 2 != 0 and frame_count > 1:
                continue

            # Process frame
            # 1. Detect traffic lights
            light_state = self.detect_traffic_light(frame)

            # 2. Detect vehicles
            detections = self.detect_vehicles(frame)

            # 3. Update vehicle tracks
            self.update_tracks(detections)

            # 4. Detect violations
            red_light, wrong_side, crash = self.detect_violations(frame)

            # 5. Draw annotations
            annotated_frame = self.draw_annotations(frame.copy())

            # 6. Write to output video if needed
            if output_video_path:
                out.write(annotated_frame)

            # 7. Display if requested
            if display_output and frame_count % 10 == 0:  # Show every 10th frame for performance
                # Convert BGR to RGB for display
                rgb_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
                plt.figure(figsize=(12, 8))
                plt.imshow(rgb_frame)
                plt.axis('off')
                plt.show()

            # Print progress
            if frame_count % 30 == 0:
                fps = frame_count / (time.time() - start_time)
                print(f"Processed {frame_count} frames. FPS: {fps:.2f}")
                print(f"Traffic light: {light_state}, Direction: {self.traffic_direction}")
                print(f"Violations - Red Light: {len(red_light)}, Wrong Side: {len(wrong_side)}, Crash: {len(crash)}")

        # Clean up
        self.cap.release()
        if output_video_path:
            out.release()

        # Print final stats
        elapsed_time = time.time() - start_time
        print(f"Video processing complete. Processed {frame_count} frames in {elapsed_time:.2f} seconds.")
        print(f"Detected violations have been saved to {self.output_dir}")

        # Return paths for convenience
        return self.screenshot_dir, self.log_file

# Utility function for Google Colab
def upload_and_process_video():
    """Upload a video file and process it with the violation detector."""
    print("Please upload a video file...")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return

    video_path = list(uploaded.keys())[0]
    print(f"Processing video: {video_path}")

    # Process the video
    detector = TrafficViolationDetector(video_path)
    screenshots_dir, log_file = detector.process_video(display_output=True, output_video_path="processed_video.mp4")

    # Display some results
    print("\nViolation Statistics:")
    violation_types = []
    with open(log_file, 'r') as f:
        next(f)  # Skip header
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 2:
                violation_types.append(parts[1])

    # Count violations by type
    from collections import Counter
    violation_counts = Counter(violation_types)

    print("\nViolation Summary:")
    for violation, count in violation_counts.items():
        print(f"{violation}: {count}")

    # Offer to download processed video
    files.download("processed_video.mp4")
    print("\nYou can also download the processed video using the link above.")

# Usage example
if __name__ == "__main__":
    upload_and_process_video()

In [ ]:
def license_plate_reader():

    # Open the video file
    vid = cv2.VideoCapture('/traffic_video_original.mp4')
    # Create detector object
    detector = LineDetector()

    # Loop through each frame in the video
    while True:
        # Read frame
        ret, frame = vid.read()

        # Break if frame is not returned
        if not ret:
            break

        # Assuming rect is the rectangle where the traffic light is located
        rect = (1700, 40, 100, 250)

        # Detect traffic light color
        frame, color = detect_traffic_light_color(frame, rect)

        # Detect white line
        frame, mask_line = detector.detect_white_line(frame, color)

        # Process the frame if the light is red
        if color == 'red':
            # Extract license plate
            frame, license_plate_images = extract_license_plate(frame, mask_line)

            # Process each detected license plate
            for license_plate_image in license_plate_images:
                # Apply OCR to the license plate image
                text = apply_ocr_to_image(license_plate_image)

                # Add the detected license plate to the list if it matches the pattern and is not already in the list
                if text is not None and re.match("^[A-Z]{2}\s[0-9]{3,4}$", text) and text not in penalized_texts:
                    penalized_texts.append(text)
                    print(f"\nFined license plate: {text}")

                    # Plot the license plate image
                    plt.figure()
                    plt.imshow(license_plate_image, cmap='gray')
                    plt.axis('off')
                    plt.show()

                    # Update the database with the license plate violation
                    update_database_with_violation(text, DB_HOST, DB_USER, DB_PASSWORD, DB_NAME)

        # Draw the penalized text onto the frame if there is any
        if penalized_texts:
            draw_penalized_text(frame)

        # Display the frame
        cv2.imshow('frame', frame)

        # Break if ESC key is pressed (uncomment the following line when running on a local system with GUI support)
        if cv2.waitKey(1) == 27:
            break

    # Release the video
    vid.release()

    # Close all OpenCV windows (uncomment the following line when running on a local system with GUI support)
    cv2.destroyAllWindows()

In [ ]:
import requests
import cv2
# Download the trained Haar Cascade from the GitHub repository
url = "https://raw.githubusercontent.com/FarzadNekouee/Traffic-Violation-Detection/master/haarcascade_russian_plate_number.xml"
response = requests.get(url)

with open('haarcascade_russian_plate_number.xml', 'wb') as file:
    file.write(response.content)

# Load the trained Haar Cascade
license_plate_cascade = cv2.CascadeClassifier('haarcascade_russian_plate_number.xml')

# Create a list to store unique penalized license plate texts
penalized_texts = []

In [ ]:
from google.colab import drive
drive.mount('/content/drive')